In [ ]:
import os, glob

ROOT = r"data/Fish Freshness-eyes-skin"
SPLITS = ["train","valid","test"]
NUM_CLASSES = 6

def ok_line(parts):
    if len(parts)!=5: return False
    try:
        c = int(float(parts[0])); vals = list(map(float, parts[1:]))
    except: return False
    if not (0 <= c < NUM_CLASSES): return False
    x,y,w,h = vals
    return 0 < w <= 1 and 0 < h <= 1 and 0 <= x <= 1 and 0 <= y <= 1

bad = []
for sp in SPLITS:
    for lf in glob.glob(os.path.join(ROOT, sp, "labels", "*.txt")):
        with open(lf, "r", encoding="utf-8", errors="ignore") as f:
            lines = [ln.strip() for ln in f if ln.strip()]
        good = [ln for ln in lines if ok_line(ln.split())]
        if len(good) != len(lines):
            bad.append(lf)
            # keep only good lines; if none, leave empty file
            with open(lf, "w", encoding="utf-8") as f:
                for ln in good: f.write(ln+"\n")
print("fixed/flagged:", len(bad))


fixed/flagged: 17


In [1]:
# D:\Projects\MATSYA\codes\yolo_to_crops.py
import os, cv2, glob
from pathlib import Path
import yaml

# ---- CONFIG ----
DATA_ROOT = r"data/Fish Freshness-eyes-skin"  # has train/valid/test
OUT_ROOT  = r"freshness_crops_cls"       # will be created
PAD = 0.12            # 12% padding around bbox
MIN_SIDE = 16         # skip tiny crops
# If you have data.yaml with names, we'll read it; else fallback:
DATA_YAML = os.path.join(DATA_ROOT, "data.yaml")
FALLBACK_NAMES = [
    "Fresh-Eye","Fresh-Skin","NonFresh-Eye","NonFresh-Skin","VeryFresh-Eye","VeryFresh-Skin"
]
# ---------------

def load_names():
    if os.path.isfile(DATA_YAML):
        with open(DATA_YAML, "r", encoding="utf-8") as f:
            y = yaml.safe_load(f)
        names = y.get("names", FALLBACK_NAMES)
        # names can be dict or list
        if isinstance(names, dict):
            names = [names[k] for k in sorted(names.keys(), key=lambda x:int(x))]
        return names
    return FALLBACK_NAMES

NAMES = load_names()
os.makedirs(OUT_ROOT, exist_ok=True)

def yolo_to_xyxy(lbl, W, H):
    # lbl: [cls, x, y, w, h] normalized
    c, x, y, w, h = lbl
    x, y, w, h = float(x)*W, float(y)*H, float(w)*W, float(h)*H
    x1 = max(0, int(x - w/2)); y1 = max(0, int(y - h/2))
    x2 = min(W-1, int(x + w/2)); y2 = min(H-1, int(y + h/2))
    return int(c), x1, y1, x2, y2

def pad_box(x1,y1,x2,y2,W,H,ratio):
    bw, bh = x2-x1, y2-y1
    px, py = int(bw*ratio), int(bh*ratio)
    return max(0,x1-px), max(0,y1-py), min(W-1,x2+px), min(H-1,y2+py)

def process_split(split):
    img_dir = os.path.join(DATA_ROOT, split, "images")
    lbl_dir = os.path.join(DATA_ROOT, split, "labels")
    paths = glob.glob(os.path.join(img_dir, "*.*"))
    for ip in paths:
        stem = Path(ip).stem
        lp = os.path.join(lbl_dir, stem + ".txt")
        if not os.path.isfile(lp):
            continue
        im = cv2.imread(ip)
        if im is None:
            continue
        H, W = im.shape[:2]
        with open(lp, "r", encoding="utf-8", errors="ignore") as f:
            lines = [ln.strip().split() for ln in f if ln.strip()]
        for k, parts in enumerate(lines):
            if len(parts) != 5:
                continue
            c, x1, y1, x2, y2 = yolo_to_xyxy(parts, W, H)
            x1,y1,x2,y2 = pad_box(x1,y1,x2,y2,W,H,PAD)
            if x2-x1 < MIN_SIDE or y2-y1 < MIN_SIDE:
                continue
            crop = im[y1:y2, x1:x2]
            cls_name = NAMES[int(c)]
            out_dir = os.path.join(OUT_ROOT, split, cls_name)
            os.makedirs(out_dir, exist_ok=True)
            out_path = os.path.join(out_dir, f"{stem}_{k}.jpg")
            cv2.imwrite(out_path, crop)

for sp in ("train","valid","test"):
    process_split(sp)
print("Done. Crops at:", OUT_ROOT)


Done. Crops at: freshness_crops_cls


In [2]:
# D:\Projects\MATSYA\codes\train_mobilenet_freshness_cls.py
import os, random, torch, torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

DATA = r"freshness_crops_cls"  # from step 1
OUT  = r"models/freshness_cls"
IMG=224; BATCH=16; EPOCHS=30; SEED=42; NUM_WORKERS=2; LABEL_SMOOTH=0.05

os.makedirs(OUT, exist_ok=True)
device="cuda" if torch.cuda.is_available() else "cpu"
if device=="cuda":
    torch.backends.cudnn.benchmark=True
    torch.set_float32_matmul_precision("high")
torch.manual_seed(SEED); random.seed(SEED)

# datasets from split subfolders
tfm_tr = transforms.Compose([
    transforms.Resize((IMG,IMG)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.25,0.25,0.25,0.05),
    transforms.RandomAffine(12, translate=(0.08,0.08), scale=(0.9,1.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
tfm_val = transforms.Compose([
    transforms.Resize((IMG,IMG)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_ds = datasets.ImageFolder(os.path.join(DATA,"train"), transform=tfm_tr)
val_ds   = datasets.ImageFolder(os.path.join(DATA,"valid"), transform=tfm_val)
test_ds  = datasets.ImageFolder(os.path.join(DATA,"test"),  transform=tfm_val)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device=="cuda"))
val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device=="cuda"))
test_loader  = DataLoader(test_ds, batch_size=BATCH, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device=="cuda"))

num_classes = len(train_ds.classes)
print("Classes:", train_ds.classes)

model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
model = model.to(device, memory_format=torch.channels_last)

ce = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = CosineAnnealingLR(opt, T_max=EPOCHS*len(train_loader))
scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))

def run_epoch(loader, train=True):
    model.train(train)
    tot=0; cor=0; loss_sum=0.0
    for x,y in loader:
        x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        y = y.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(device=="cuda")):
            out = model(x)
            loss = ce(out,y)
        if train:
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
        loss_sum += loss.item()*y.size(0)
        pred = out.argmax(1); cor += (pred==y).sum().item(); tot += y.size(0)
    return loss_sum/tot, cor/tot

best = 0; best_path = os.path.join(OUT, "mobilenet_freshness_6cls.pt")
for e in range(1, EPOCHS+1):
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    print(f"[{e}] train_loss={tr[0]:.4f} acc={tr[1]:.3f} | val_loss={va[0]:.4f} acc={va[1]:.3f}")
    if va[1] > best:
        best = va[1]
        torch.save({"model":model.state_dict(), "classes":train_ds.classes}, best_path)
        print("saved:", best_path, "acc:", best)

# test accuracy
@torch.no_grad()
def eval_loader(loader):
    model.eval()
    tot=0; cor=0
    for x,y in loader:
        x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        y = y.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(device=="cuda")):
            out = model(x)
        pred = out.argmax(1); cor += (pred==y).sum().item(); tot += y.size(0)
    return cor/tot

test_acc = eval_loader(test_loader)
print("test acc:", test_acc)


Classes: ['Fresh-Eye', 'Fresh-Skin', 'NonFresh-Eye', 'NonFresh-Skin', 'VeryFresh-Eye', 'VeryFresh-Skin']


C:\Users\rupes\AppData\Local\Temp\ipykernel_7264\273837510.py:54: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))
C:\Users\rupes\AppData\Local\Temp\ipykernel_7264\273837510.py:62: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=="cuda")):
d:\Projects\environments\pytorch-env\Lib\site-packages\torch\optim\lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


[1] train_loss=0.6060 acc=0.840 | val_loss=0.5590 acc=0.857
saved: models/freshness_cls\mobilenet_freshness_6cls.pt acc: 0.8574468085106383
[2] train_loss=0.4296 acc=0.923 | val_loss=0.4158 acc=0.921
saved: models/freshness_cls\mobilenet_freshness_6cls.pt acc: 0.9212765957446809
[3] train_loss=0.3741 acc=0.949 | val_loss=0.3865 acc=0.938
saved: models/freshness_cls\mobilenet_freshness_6cls.pt acc: 0.9382978723404255
[4] train_loss=0.3398 acc=0.965 | val_loss=0.3797 acc=0.946
saved: models/freshness_cls\mobilenet_freshness_6cls.pt acc: 0.9457446808510638
[5] train_loss=0.3166 acc=0.974 | val_loss=0.3971 acc=0.943
[6] train_loss=0.3039 acc=0.980 | val_loss=0.3899 acc=0.945
[7] train_loss=0.2951 acc=0.983 | val_loss=0.3881 acc=0.946
[8] train_loss=0.2854 acc=0.987 | val_loss=0.3608 acc=0.954
saved: models/freshness_cls\mobilenet_freshness_6cls.pt acc: 0.9542553191489361
[9] train_loss=0.2774 acc=0.990 | val_loss=0.3620 acc=0.951
[10] train_loss=0.2742 acc=0.991 | val_loss=0.3605 acc=0.957

C:\Users\rupes\AppData\Local\Temp\ipykernel_7264\273837510.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=="cuda")):


test acc: 0.9637139807897546


In [3]:
# Assuming you have already trained your model and loaded the best weights
# Load the best model from the saved .pt file
best_path = os.path.join(OUT, "mobilenet_freshness_6cls.pt")
state_dict = torch.load(best_path, map_location=device)["model"]
model.load_state_dict(state_dict)
model.eval() # Set the model to evaluation mode

# Define the path for the ONNX file
onnx_path = os.path.join(OUT, "mobilenet_freshness_6cls.onnx")

# Create a dummy input tensor
# The size is [batch_size, channels, height, width]
dummy_input = torch.randn(1, 3, IMG, IMG, device=device)

# Export the model to ONNX
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    verbose=False,
    input_names=['input'],
    output_names=['output'],
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print(f"Model successfully saved to ONNX at: {onnx_path}")

Model successfully saved to ONNX at: models/freshness_cls\mobilenet_freshness_6cls.onnx


In [4]:

# 🔧 Paths
FRESH_CLS_CKPT = r"models/freshness_cls/mobilenet_freshness_6cls.pt"
FRESH_CLS_VAL_DIR = r"data/freshness_crops_cls/valid"

import os, json, random, time, math
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42); random.seed(42); np.random.seed(42)

IMG=224; BATCH=32
tfm = transforms.Compose([
    transforms.Resize((IMG,IMG)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

val_ds = datasets.ImageFolder(FRESH_CLS_VAL_DIR, transform=tfm)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)
classes = val_ds.classes; num_classes=len(classes)
print("Classes:", classes, "| Val size:", len(val_ds))

ckpt = torch.load(FRESH_CLS_CKPT, map_location=device)
model = models.mobilenet_v3_small(weights=None)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
model.load_state_dict(ckpt.get("model", ckpt))
model.to(device).eval()

y_true=[]; y_pred=[]; y_prob=[]
with torch.no_grad():
    for x,y in val_loader:
        x = x.to(device)
        out = model(x)
        prob = torch.softmax(out, dim=1).cpu().numpy()
        y_prob.append(prob)
        y_true.extend(y.numpy().tolist())
        y_pred.extend(np.argmax(prob, axis=1).tolist())
y_true = np.array(y_true); y_pred = np.array(y_pred); y_prob = np.concatenate(y_prob, axis=0)
acc = (y_true==y_pred).mean()
print(f"Val accuracy: {acc:.4f}")


Classes: ['Fresh-Eye', 'Fresh-Skin', 'NonFresh-Eye', 'NonFresh-Skin', 'VeryFresh-Eye', 'VeryFresh-Skin'] | Val size: 940
Val accuracy: 0.9691
